# CRGCN trên REES46 Full — Kaggle (offline)

Chạy **offline trên Kaggle** (không cần bật Internet):
- **Data**: đọc từ Kaggle Dataset gắn vào `/kaggle/input/...` (cấu trúc giống `rees46-full-temporal`). Cell §2 tự dò đường dẫn.
- **Mã CRGCN**: nhúng sẵn bằng `%%writefile` (không `git clone`).
- **PyG** (`torch_geometric/torch_scatter/torch_sparse`): dùng bản có sẵn trong image Kaggle; nếu thiếu sẽ thử cài offline từ dataset wheels (tên chứa `pyg`/`wheel`).
- **W&B**: tắt (`WANDB_MODE=disabled`); checkpoint lưu vào `/kaggle/working/`.

> Yêu cầu: bật **GPU** ở Settings → Accelerator (không phải Internet).

## 1. Chuẩn bị môi trường + nhúng mã CRGCN (offline)

In [ ]:
# Offline: notebook KHÔNG cần torch_geometric/torch_scatter/torch_sparse nữa.
# gcn_conv.py đã được viết lại bằng pure PyTorch (xem cell %%writefile bên dưới).
import torch, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', np.__version__)

torch 2.10.0+cu128 | cuda True | numpy 2.4.6


In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# @File  : utils.py
# @Author: yanms
# @Date  : 2021/10/23 11:26
# @Desc  :
import torch
import torch.nn as nn


class BPRLoss(nn.Module):
    def __init__(self):
        super(BPRLoss, self).__init__()
        self.gamma = 1e-10

    def forward(self, p_score, n_score):
        loss = -torch.log(self.gamma + torch.sigmoid(p_score - n_score))
        loss = loss.mean()

        return loss


class EmbLoss(nn.Module):
    """ EmbLoss, regularization on embeddings

    """

    def __init__(self, norm=2):
        super(EmbLoss, self).__init__()
        self.norm = norm

    def forward(self, *embeddings):
        emb_loss = torch.zeros(1).to(embeddings[-1].device)
        for embedding in embeddings:
            emb_loss += torch.norm(embedding, p=self.norm)
        emb_loss /= embeddings[-1].shape[0]
        return emb_loss



Overwriting utils.py


In [ ]:
%%writefile gcn_conv.py
# -*- coding: utf-8 -*-
# gcn_conv.py — pure-PyTorch reimplement của CRGCN GCNConv (LightGCN-style).
# Bỏ phụ thuộc torch_geometric / torch_scatter / torch_sparse để chạy OFFLINE.
# Toán học giữ nguyên: out = D^-1/2 · A · D^-1/2 · x (+ bias), không có linear
# (bản gốc CRGCN đã comment self.lin).
import torch
from torch import Tensor
from torch.nn import Parameter


class GCNConv(torch.nn.Module):
    """GCN propagation chuẩn hóa đối xứng, không có phép biến đổi tuyến tính.
    Giữ nguyên chữ ký __init__ để GraphEncoder dùng không đổi.
    """

    def __init__(self, in_channels, out_channels,
                 improved=False, cached=False,
                 add_self_loops=True, normalize=True, bias=True, **kwargs):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops
        self.normalize = normalize
        if bias:
            self.bias = Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter('bias', None)

    def reset_parameters(self):
        if self.bias is not None:
            torch.nn.init.zeros_(self.bias)

    def forward(self, x: Tensor, edge_index: Tensor, edge_weight=None) -> Tensor:
        n = x.size(0)
        row, col = edge_index[0], edge_index[1]

        if self.add_self_loops:
            loop = torch.arange(n, device=edge_index.device)
            row = torch.cat([row, loop])
            col = torch.cat([col, loop])

        if edge_weight is None:
            edge_weight = torch.ones(row.size(0), device=x.device, dtype=x.dtype)

        if self.normalize:
            deg = torch.zeros(n, device=x.device, dtype=x.dtype)
            deg.scatter_add_(0, col, edge_weight)
            deg_inv_sqrt = deg.pow(-0.5)
            deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0.0)
            edge_weight = deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]

        # out[col] += edge_weight * x[row]  (aggr='add', flow=source_to_target)
        out = torch.zeros_like(x)
        out.index_add_(0, col, x[row] * edge_weight.unsqueeze(-1))

        if self.bias is not None:
            out = out + self.bias
        return out

    def __repr__(self):
        return '{}({}, {})'.format(self.__class__.__name__,
                                   self.in_channels, self.out_channels)

Overwriting gcn_conv.py


In [ ]:
%%writefile model_cascade.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# @File  : model.py
# @Author: yanms
# @Date  : 2021/11/1 16:16
# @Desc  : CRGCN
import os.path

import torch
import torch.nn as nn
import torch.nn.functional as F

from gcn_conv import GCNConv
from utils import BPRLoss, EmbLoss


class GraphEncoder(nn.Module):
    def __init__(self, layers, hidden_dim, dropout):
        super(GraphEncoder, self).__init__()
        self.gnn_layers = nn.ModuleList(
            [GCNConv(hidden_dim, hidden_dim, add_self_loops=False, cached=False) for i in range(layers)])
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, edge_index):

        for i in range(len(self.gnn_layers)):
            x = self.gnn_layers[i](x=x, edge_index=edge_index)
            # x = self.dropout(x)
        return x


class CRGCN(nn.Module):
    def __init__(self, args, dataset):
        super(CRGCN, self).__init__()

        self.device = args.device
        self.layers = args.layers
        self.node_dropout = args.node_dropout
        self.message_dropout = nn.Dropout(p=args.message_dropout)
        self.n_users = dataset.user_count
        self.n_items = dataset.item_count
        self.edge_index = dataset.edge_index
        self.behaviors = args.behaviors
        self.embedding_size = args.embedding_size
        self.user_embedding = nn.Embedding(self.n_users + 1, self.embedding_size, padding_idx=0)
        self.item_embedding = nn.Embedding(self.n_items + 1, self.embedding_size, padding_idx=0)
        self.Graph_encoder = nn.ModuleDict({
            behavior: GraphEncoder(self.layers[index], self.embedding_size, self.node_dropout) for index, behavior in enumerate(self.behaviors)
        })


        self.reg_weight = args.reg_weight
        self.bpr_loss = BPRLoss()
        self.emb_loss = EmbLoss()

        self.model_path = args.model_path
        self.check_point = args.check_point
        self.if_load_model = args.if_load_model

        self.storage_all_embeddings = None

        self.apply(self._init_weights)

        self._load_model()

    def _init_weights(self, module):

        if isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight.data)


    def _load_model(self):
        if self.if_load_model:
            parameters = torch.load(os.path.join(self.model_path, self.check_point))
            self.load_state_dict(parameters, strict=False)


    def gcn_propagate(self):
        """
        gcn propagate in each behavior
        """
        all_embeddings = {}
        total_embeddings = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        for behavior in self.behaviors:
            layer_embeddings = total_embeddings
            indices = self.edge_index[behavior].to(self.device)
            layer_embeddings = self.Graph_encoder[behavior](layer_embeddings, indices)
            layer_embeddings = F.normalize(layer_embeddings, dim=-1)
            total_embeddings = layer_embeddings + total_embeddings
            all_embeddings[behavior] = total_embeddings
        return all_embeddings

    def forward(self, batch_data):
        self.storage_all_embeddings = None

        all_embeddings = self.gcn_propagate()
        total_loss = 0
        for index, behavior in enumerate(self.behaviors):
            data = batch_data[:, index]
            users = data[:, 0].long()
            items = data[:, 1:].long()
            user_all_embedding, item_all_embedding = torch.split(all_embeddings[behavior], [self.n_users + 1, self.n_items + 1])

            user_feature = user_all_embedding[users.view(-1, 1)].expand(-1, items.shape[1], -1)
            item_feature = item_all_embedding[items]
            # user_feature, item_feature = self.message_dropout(user_feature), self.message_dropout(item_feature)

            scores = torch.sum(user_feature * item_feature, dim=2)
            total_loss += self.bpr_loss(scores[:, 0], scores[:, 1])
        total_loss = total_loss + self.reg_weight * self.emb_loss(self.user_embedding.weight, self.item_embedding.weight)

        return total_loss

    def full_predict(self, users):
        if self.storage_all_embeddings is None:
            self.storage_all_embeddings = self.gcn_propagate()

        user_embedding, item_embedding = torch.split(self.storage_all_embeddings[self.behaviors[-1]], [self.n_users + 1, self.n_items + 1])
        user_emb = user_embedding[users.long()]
        scores = torch.matmul(user_emb, item_embedding.transpose(0, 1))
        return scores



Overwriting model_cascade.py


In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
from model_cascade import CRGCN
print('[ok] CRGCN nhúng sẵn — import thành công')

[ok] CRGCN nhúng sẵn — import thành công


## 2. Tìm dataset trong `/kaggle/input`

In [ ]:
import os, glob

# Tự dò thư mục data (mốc: purchase_train_src.npy) trong mọi dataset đã gắn
_hits = glob.glob('/kaggle/input/**/purchase_train_src.npy', recursive=True)
if not _hits:
    raise FileNotFoundError(
        'Không thấy purchase_train_src.npy trong /kaggle/input. '
        'Hãy bấm "Add Input" và chọn dataset rees46-full-temporal.')
DATA_DIR = os.path.dirname(_hits[0]) + '/'
print('DATA_DIR =', DATA_DIR)
print('files:', sorted(os.listdir(DATA_DIR))[:20])

DATA_DIR = /kaggle/input/datasets/bluet52hzzz/rees46-full-temporal/rees46-full-temporal/
files: ['.cache', '.gitattributes', 'SUBSAMPLE_MANIFEST.json', 'candidate_item_idx.npy', 'cart_train_dst.npy', 'cart_train_src.npy', 'cart_train_ts.npy', 'cart_trainval_dst.npy', 'cart_trainval_src.npy', 'cart_trainval_ts.npy', 'graph', 'item_is_cold.npy', 'node_counts.json', 'node_mappings', 'purchase_train_dst.npy', 'purchase_train_src.npy', 'purchase_train_ts.npy', 'purchase_trainval_dst.npy', 'purchase_trainval_src.npy', 'purchase_trainval_ts.npy']


## 3. Load `training.yaml`

In [ ]:
import yaml, io, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

YAML_TEXT = """
data:
  data_dir: /kaggle/input/
  node_counts:
    product: 100775
    user: 245778
model:
  embed_dim: 64
  n_layers: 1
  dropout: 0.4
training:
  batch_size: 2048
  eval_batch_size: 1024
  epochs: 40
  lr: 1.0e-03
  l2_lambda: 1.0e-04
  seed: 42
  eval_every: 1
  patience: 5
  num_neg: 1
  device: cuda
  save_dir: /kaggle/working/checkpoints-crgcn
  max_view_triplets: 3000000
  max_cart_triplets: 3868050
  max_purchase_triplets: 0
evaluation:
  primary_metric: NDCG@20
  ks: [1, 5, 10, 20, 50]
wandb:
  project: rees46-full-recsys
  run_name: crgcn-full-kaggle
  artifact_name: crgcn-full-ckpt
"""

yaml_path = os.path.join(DATA_DIR, 'training.yaml')
if os.path.isfile(yaml_path):
    with open(yaml_path) as f:
        CFG = yaml.safe_load(f)
else:
    CFG = yaml.safe_load(io.StringIO(YAML_TEXT))

CFG.setdefault('data', {})['data_dir'] = DATA_DIR

CFG.setdefault('training', {})
T = CFG['training']
T.setdefault('save_dir', '/kaggle/working/checkpoints-crgcn')
T.setdefault('num_neg', 1)
T.setdefault('max_view_triplets', 3000000)
T.setdefault('max_cart_triplets', 3868050)
T.setdefault('max_purchase_triplets', 0)
T.setdefault('eval_batch_size', 1024)

CFG.setdefault('model', {})
if CFG['model'].get('embed_dim', 64) > 64:
    print('[adjust] embed_dim', CFG['model']['embed_dim'], '-> 64')
    CFG['model']['embed_dim'] = 64
if CFG['model'].get('n_layers', 1) > 2:
    print('[adjust] n_layers', CFG['model']['n_layers'], '-> 1')
    CFG['model']['n_layers'] = 1
if CFG['training'].get('batch_size', 2048) > 2048:
    CFG['training']['batch_size'] = 2048
if CFG['training'].get('max_view_triplets', 0) > 3000000:
    CFG['training']['max_view_triplets'] = 3000000
if CFG['training'].get('max_cart_triplets', 0) > 3868050:
    CFG['training']['max_cart_triplets'] = 3868050

os.makedirs(CFG['training']['save_dir'], exist_ok=True)
print(yaml.safe_dump(CFG, sort_keys=False))

data:
  data_dir: /kaggle/input/datasets/bluet52hzzz/rees46-full-temporal/rees46-full-temporal/
  node_counts:
    product: 100775
    user: 245778
model:
  embed_dim: 64
  n_layers: 1
  dropout: 0.4
training:
  batch_size: 2048
  eval_batch_size: 1024
  epochs: 40
  lr: 0.001
  l2_lambda: 0.0001
  seed: 42
  eval_every: 1
  patience: 5
  num_neg: 1
  device: cuda
  save_dir: /kaggle/working/checkpoints-crgcn
  max_view_triplets: 3000000
  max_cart_triplets: 3868050
  max_purchase_triplets: 0
evaluation:
  primary_metric: NDCG@20
  ks:
  - 1
  - 5
  - 10
  - 20
  - 50
wandb:
  project: rees46-full-recsys
  run_name: crgcn-full-kaggle
  artifact_name: crgcn-full-ckpt



## 3b. Weights & Biases — TẮT (offline)

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'   # offline: mọi lệnh wandb thành no-op, không gọi mạng
import wandb

WB_CFG = CFG.get('wandb', {}) or {}
run = wandb.init(
    project=WB_CFG.get('project', 'rees46-full-recsys'),
    name=WB_CFG.get('run_name', 'crgcn-full-kaggle'),
    config=CFG,
    mode='disabled',
)
ARTIFACT_NAME = WB_CFG.get('artifact_name', 'crgcn-full-ckpt')
print('W&B: disabled (offline) — checkpoint lưu local ở', CFG['training']['save_dir'])

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: ERROR Unable to load kaggle notebook.


W&B: disabled (offline) — checkpoint lưu local ở /kaggle/working/checkpoints-crgcn


## 4. Load REES46 Full data → adapter cho CRGCN

CRGCN dùng index 1-based (padding_idx=0); REES46 dùng 0-based → shift `+1`.

In [ ]:
import numpy as np, pickle, torch, json, os
from types import SimpleNamespace

DATA = CFG['data']['data_dir']
N_USERS = CFG['data']['node_counts']['user']
N_ITEMS = CFG['data']['node_counts']['product']

BEHAVIORS = ['view', 'cart', 'purchase']
CAPS = {
    'view':     CFG['training'].get('max_view_triplets', 1_500_000),
    'cart':     CFG['training'].get('max_cart_triplets', 1_500_000),
    'purchase': CFG['training'].get('max_purchase_triplets', 0),
}
RNG = np.random.default_rng(CFG['training']['seed'])

def load_edges(name):
    src = np.load(os.path.join(DATA, f'{name}_train_src.npy')).astype(np.int64)
    dst = np.load(os.path.join(DATA, f'{name}_train_dst.npy')).astype(np.int64)
    cap = CAPS.get(name, 0)
    if cap and len(src) > cap:
        sel = RNG.choice(len(src), size=cap, replace=False)
        src, dst = src[sel], dst[sel]
        print(f'  {name}: subsampled to {cap:,} edges')
    return src, dst

edges = {b: load_edges(b) for b in BEHAVIORS}
for b, (s, d) in edges.items():
    print(f'{b}: edges={len(s):,}  u_max={s.max()}  i_max={d.max()}')

with open(os.path.join(DATA, 'train_mask_purchase_only.pkl'), 'rb') as f:
    TRAIN_MASK = pickle.load(f)
with open(os.path.join(DATA, 'val_ground_truth.pkl'), 'rb') as f:
    VAL_GT = pickle.load(f)
with open(os.path.join(DATA, 'test_ground_truth.pkl'), 'rb') as f:
    TEST_GT = pickle.load(f)
CAND = np.load(os.path.join(DATA, 'candidate_item_idx.npy')).astype(np.int64)
print(f'val users={len(VAL_GT):,}  test users={len(TEST_GT):,}  candidates={len(CAND):,}')

  view: subsampled to 3,000,000 edges
view: edges=3,000,000  u_max=184076  i_max=85665
cart: edges=970,492  u_max=184076  i_max=85665
purchase: edges=612,517  u_max=184094  i_max=85665
val users=16,189  test users=8,104  candidates=100,775


In [ ]:
class BPATMPDataset:
    def __init__(self, n_users, n_items, edges_dict, behaviors):
        self.user_count = n_users
        self.item_count = n_items
        self.behaviors = behaviors
        self.edge_index = {}
        self.behavior_dict = {b: {} for b in behaviors}
        self.behavior_dict['all'] = {}
        self.user_behaviour_degree = []
        for b in behaviors:
            src, dst = edges_dict[b]
            u = torch.from_numpy(src + 1).long()
            i = torch.from_numpy(dst + 1).long()
            deg = torch.zeros(n_users + 1, dtype=torch.float32)
            deg.scatter_add_(0, u, torch.ones_like(u, dtype=torch.float32))
            self.user_behaviour_degree.append(deg.view(-1, 1))
            col = i + (n_users + 1)
            row_all = torch.cat([u, col])
            col_all = torch.cat([col, u])
            self.edge_index[b] = torch.stack([row_all, col_all])
            bd = {}
            for uu, ii in zip(src.tolist(), dst.tolist()):
                bd.setdefault(uu + 1, []).append(ii + 1)
            self.behavior_dict[b] = bd
        all_d = {}
        for b in behaviors:
            for u, lst in self.behavior_dict[b].items():
                all_d.setdefault(u, set()).update(lst)
        self.behavior_dict['all'] = {u: np.array(sorted(s), dtype=np.int64) for u, s in all_d.items()}
        self.user_behaviour_degree = torch.cat(self.user_behaviour_degree, dim=1)

dataset = BPATMPDataset(N_USERS, N_ITEMS, edges, BEHAVIORS)
print('Built dataset. edge_index keys:', list(dataset.edge_index.keys()))

Built dataset. edge_index keys: ['view', 'cart', 'purchase']


## 5. Khởi tạo model CRGCN

In [ ]:
import random, numpy as np, torch
SEED = CFG['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from model_cascade import CRGCN

device = CFG['training']['device'] if torch.cuda.is_available() else 'cpu'
args = SimpleNamespace(
    device=device,
    layers=[CFG['model']['n_layers']] * len(BEHAVIORS),
    node_dropout=CFG['model']['dropout'],
    message_dropout=CFG['model']['dropout'],
    embedding_size=CFG['model']['embed_dim'],
    behaviors=BEHAVIORS,
    reg_weight=CFG['training']['l2_lambda'],
    model_path=CFG['training']['save_dir'],
    check_point='',
    if_load_model=False,
)
model = CRGCN(args, dataset).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model on {device} | params = {n_params/1e6:.2f}M')

model on cuda | params = 22.18M


## 6. Evaluator (port từ `evaluator.py`)
Full-rank, tiled, mask exclusion bằng `train_mask_purchase_only`, HR@k & NDCG@k.

In [ ]:
@torch.no_grad()
def evaluate(model, gt_dict, exclude_dict, ks=(1,5,10,20,50), user_batch=512, item_tile=16384):
    model.eval()
    model.storage_all_embeddings = None
    max_k = max(ks)
    dev = next(model.parameters()).device
    dtype = torch.float16 if dev.type == 'cuda' else torch.float32

    model.storage_all_embeddings = model.gcn_propagate()
    last_b = model.behaviors[-1]
    u_all, i_all = torch.split(
        model.storage_all_embeddings[last_b],
        [model.n_users + 1, model.n_items + 1],
    )
    item_embs = i_all[1:N_ITEMS + 1].to(dtype)
    user_all = u_all.to(dtype)

    eval_uids = np.array(sorted(gt_dict.keys()), dtype=np.int64)
    n_eval = len(eval_uids)

    gt_lists = [np.asarray(gt_dict[u], dtype=np.int64).reshape(-1) for u in eval_uids]
    max_pos = max(len(g) for g in gt_lists)
    gt_pad = torch.full((n_eval, max_pos), -1, dtype=torch.long, device=dev)
    gt_cnt = torch.zeros(n_eval, dtype=torch.long, device=dev)
    for r, g in enumerate(gt_lists):
        gt_cnt[r] = len(g)
        gt_pad[r, :len(g)] = torch.as_tensor(g, device=dev)

    uid2pos = {int(u): i for i, u in enumerate(eval_uids.tolist())}
    er, ec = [], []
    for u, items in exclude_dict.items():
        p = uid2pos.get(int(u))
        if p is None or len(items) == 0: continue
        items = np.asarray(items, dtype=np.int64).reshape(-1)
        er.extend([p] * len(items)); ec.extend(items.tolist())
    er = torch.as_tensor(er, dtype=torch.long, device=dev)
    ec = torch.as_tensor(ec, dtype=torch.long, device=dev)

    ndcg_w = 1.0 / torch.log2(torch.arange(1, max_k + 1, device=dev).float() + 1.0)
    sums = {f'{m}@{k}': 0.0 for m in ('HR', 'NDCG') for k in ks}

    for us in range(0, n_eval, user_batch):
        ue = min(us + user_batch, n_eval)
        B = ue - us
        uids_b = torch.as_tensor(eval_uids[us:ue] + 1, dtype=torch.long, device=dev)
        u_emb = user_all[uids_b]
        top_v = torch.full((B, max_k), float('-inf'), device=dev, dtype=dtype)
        top_i = torch.full((B, max_k), -1, device=dev, dtype=torch.long)
        inb = (er >= us) & (er < ue)
        sr, sc = er[inb] - us, ec[inb]
        n_items = item_embs.size(0)
        for ts in range(0, n_items, item_tile):
            te = min(ts + item_tile, n_items)
            tile = item_embs[ts:te]
            scores = u_emb @ tile.T
            tmask = (sc >= ts) & (sc < te)
            if tmask.any():
                scores[sr[tmask], sc[tmask] - ts] = float('-inf')
            k = min(max_k, scores.size(1))
            tv, ti = scores.topk(k, dim=-1)
            ti = ti + ts
            mv = torch.cat([top_v, tv], dim=-1)
            mi = torch.cat([top_i, ti], dim=-1)
            sel = mv.topk(max_k, dim=-1).indices
            top_v = mv.gather(1, sel); top_i = mi.gather(1, sel)
        gtb = gt_pad[us:ue]; cnb = gt_cnt[us:ue]
        hits = (top_i.unsqueeze(-1) == gtb.unsqueeze(1)).any(dim=-1)
        for k in ks:
            sums[f'HR@{k}'] += hits[:, :k].any(dim=-1).float().sum().item()
            dcg = (hits[:, :k].float() * ndcg_w[:k]).sum(dim=-1)
            idl = torch.minimum(cnb, torch.full_like(cnb, k))
            idcg = torch.zeros_like(dcg)
            for v in idl.unique():
                mm = idl == v
                if int(v) > 0: idcg[mm] = ndcg_w[:int(v)].sum()
            sums[f'NDCG@{k}'] += (dcg / idcg.clamp_min(1e-12)).sum().item()
    model.storage_all_embeddings = None
    return {k: v / n_eval for k, v in sums.items()}

print('evaluate() defined')

evaluate() defined


## 7. Training loop

In [ ]:
from torch.utils.data import Dataset, DataLoader
import time, gc

class BehaviorTripletDataset(Dataset):
    def __init__(self, dataset, behaviors, n_items):
        self.bd = dataset.behavior_dict
        self.behaviors = behaviors
        self.n_users = dataset.user_count
        self.n_items = n_items
    def __len__(self): return self.n_users
    def __getitem__(self, idx):
        u = idx + 1
        out = np.zeros((len(self.behaviors), 3), dtype=np.int64)
        all_pos = self.bd['all'].get(u)
        for k, b in enumerate(self.behaviors):
            lst = self.bd[b].get(u)
            if not lst: continue
            pos = lst[np.random.randint(len(lst))]
            neg = np.random.randint(1, self.n_items + 1)
            if all_pos is not None and len(all_pos) > 0:
                for _ in range(5):
                    if not np.isin(neg, all_pos): break
                    neg = np.random.randint(1, self.n_items + 1)
            out[k] = [u, pos, neg]
        return out

train_ds = BehaviorTripletDataset(dataset, BEHAVIORS, N_ITEMS)
train_loader = DataLoader(train_ds, batch_size=CFG['training']['batch_size'],
                          shuffle=True, num_workers=2, drop_last=True)

optim = torch.optim.Adam(model.parameters(), lr=CFG['training']['lr'])

best_metric = -1.0; best_epoch = -1
patience = CFG['training']['patience']; stale = 0
ks = CFG['evaluation']['ks']; primary = CFG['evaluation']['primary_metric']
save_dir = CFG['training']['save_dir']
best_ckpt_path = os.path.join(save_dir, 'best.pt')

def upload_ckpt_to_wandb(path, epoch, metric_value, aliases=None):
    art = wandb.Artifact(ARTIFACT_NAME, type='model',
                         metadata={'epoch': epoch, primary: metric_value})
    art.add_file(path)
    run.log_artifact(art, aliases=aliases or ['latest'])

def free_vram():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

for epoch in range(1, CFG['training']['epochs'] + 1):
    model.train(); t0 = time.time(); ep_loss = 0.0; nb = 0
    for batch in train_loader:
        batch = batch.to(device)
        loss = model(batch)
        optim.zero_grad(); loss.backward(); optim.step()
        ep_loss += float(loss.item()); nb += 1
        del batch, loss
    free_vram()
    avg_loss = ep_loss / max(nb, 1); dt = time.time() - t0
    print(f'epoch {epoch:03d} | loss {avg_loss:.4f} | {dt:.1f}s')
    log_payload = {'epoch': epoch, 'train/loss': avg_loss, 'train/epoch_seconds': dt}

    if epoch % CFG['training']['eval_every'] == 0:
        m = evaluate(model, VAL_GT, TRAIN_MASK, ks=ks,
                     user_batch=CFG['training']['eval_batch_size'])
        free_vram()
        print('  val:', {k: round(v, 4) for k, v in m.items()})
        for k, v in m.items(): log_payload[f'val/{k}'] = v
        cur = m[primary]
        if cur > best_metric:
            best_metric = cur; best_epoch = epoch; stale = 0
            torch.save(model.state_dict(), best_ckpt_path)
            upload_ckpt_to_wandb(best_ckpt_path, epoch, cur, aliases=['best', 'latest'])
            print(f'new best {primary}={cur:.4f} (saved + uploaded)')
            log_payload['val/best_metric'] = cur
            log_payload['val/best_epoch'] = epoch
        else:
            stale += 1
            wandb.log(log_payload, step=epoch)
            if stale >= patience:
                print(f'  early stop at epoch {epoch} (best epoch {best_epoch})')
                break
            continue
    wandb.log(log_payload, step=epoch)

print(f'\nDone. best {primary}={best_metric:.4f} @ epoch {best_epoch}')

epoch 001 | loss 1.8738 | 14.9s
  val: {'HR@1': 0.0232, 'HR@5': 0.0813, 'HR@10': 0.1157, 'HR@20': 0.157, 'HR@50': 0.2113, 'NDCG@1': 0.0232, 'NDCG@5': 0.0336, 'NDCG@10': 0.0413, 'NDCG@20': 0.0501, 'NDCG@50': 0.0601}
new best NDCG@20=0.0501 (saved + uploaded)
epoch 002 | loss 1.7870 | 14.1s
  val: {'HR@1': 0.0238, 'HR@5': 0.0814, 'HR@10': 0.1184, 'HR@20': 0.1635, 'HR@50': 0.2247, 'NDCG@1': 0.0238, 'NDCG@5': 0.0336, 'NDCG@10': 0.0417, 'NDCG@20': 0.0512, 'NDCG@50': 0.0625}
new best NDCG@20=0.0512 (saved + uploaded)
epoch 003 | loss 1.7537 | 14.1s
  val: {'HR@1': 0.024, 'HR@5': 0.0816, 'HR@10': 0.1196, 'HR@20': 0.1646, 'HR@50': 0.2302, 'NDCG@1': 0.024, 'NDCG@5': 0.0338, 'NDCG@10': 0.0419, 'NDCG@20': 0.0516, 'NDCG@50': 0.0638}
new best NDCG@20=0.0516 (saved + uploaded)
epoch 004 | loss 1.7336 | 14.1s
  val: {'HR@1': 0.0245, 'HR@5': 0.0843, 'HR@10': 0.1217, 'HR@20': 0.1669, 'HR@50': 0.2348, 'NDCG@1': 0.0245, 'NDCG@5': 0.0348, 'NDCG@10': 0.0428, 'NDCG@20': 0.0523, 'NDCG@50': 0.0651}
new best N

## 8. Đánh giá trên test set với checkpoint tốt nhất

In [ ]:
ckpt = os.path.join(save_dir, 'best.pt')
if os.path.isfile(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print('loaded', ckpt)
m_test = evaluate(model, TEST_GT, TRAIN_MASK, ks=CFG['evaluation']['ks'],
                  user_batch=CFG['training']['eval_batch_size'])
print('TEST metrics:')
for k in CFG['evaluation']['ks']:
    print(f'  HR@{k:>3} = {m_test[f"HR@{k}"]:.4f}   NDCG@{k:>3} = {m_test[f"NDCG@{k}"]:.4f}')

wandb.log({f'test/{k}': v for k, v in m_test.items()})
wandb.summary.update({f'test/{k}': v for k, v in m_test.items()})
wandb.summary['best_val_epoch'] = best_epoch
wandb.summary[f'best_val_{primary}'] = best_metric

final_art = wandb.Artifact(ARTIFACT_NAME, type='model',
                           metadata={'best_epoch': best_epoch,
                                     f'val_{primary}': best_metric,
                                     **{f'test_{k}': v for k, v in m_test.items()}})
final_art.add_file(ckpt)
run.log_artifact(final_art, aliases=['final'])
wandb.finish()

loaded /kaggle/working/checkpoints-crgcn/best.pt
TEST metrics:
  HR@  1 = 0.0133   NDCG@  1 = 0.0133
  HR@  5 = 0.0564   NDCG@  5 = 0.0247
  HR@ 10 = 0.0898   NDCG@ 10 = 0.0323
  HR@ 20 = 0.1320   NDCG@ 20 = 0.0409
  HR@ 50 = 0.1966   NDCG@ 50 = 0.0524
